# DAVID-Net Training — Kaggle

Crash-proof training with HuggingFace backup. Sessions can die — progress is safe on HF.

**Setup (one-time):**
1. Add `HF_TOKEN` to Kaggle Secrets (Settings -> Secrets -> Add)
2. Attach this repo as a Kaggle dataset (or clone in Cell 2)
3. Attach training datasets to `/kaggle/input/`
4. Select **T4 GPU** accelerator
5. Hit Run All

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers huggingface_hub pyyaml scikit-learn fastapi

import os
print("Dependencies installed.")
print(f"GPU: {os.environ.get('CUDA_VISIBLE_DEVICES', 'not set')}")

# Quick GPU check
import torch
if torch.cuda.is_available():
    print(f"CUDA: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Enable T4 in notebook settings.")

In [ ]:
# Cell 2: Clone/pull repo
REPO_DIR = "/kaggle/working/david-net-av"

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/MIHMahmudEli/david-net-av.git {REPO_DIR}

import sys
sys.path.insert(0, REPO_DIR)
print(f"Repo ready at {REPO_DIR}")

In [ ]:
# Cell 3: Load secrets (NEVER print the actual values)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception as e:
    print(f"ERROR: HF_TOKEN not found in Kaggle Secrets: {e}")
    print("Add it: Settings -> Secrets -> Add -> Name=HF_TOKEN")
    raise

# Verify token works
from huggingface_hub import HfApi
api = HfApi(token=hf_token)
print(f"HF auth OK. Logged in as: {api.whoami()['name']}")

In [ ]:
# Cell 4: Extract datasets (runs once per session, skips if already done)
import zipfile
import tarfile
from pathlib import Path

WORKING = Path("/kaggle/working")
DATA_DIR = WORKING / "data"
DATA_DIR.mkdir(exist_ok=True)

# Map dataset names to their Kaggle mount paths
DATASETS = {
    "fakeavceleb": "/kaggle/input/fakeavceleb-v1-2",
    "lav-df": "/kaggle/input/lav-df",
    "dfdc-10": "/kaggle/input/dfdc-10",
    "deepfaketimit": "/kaggle/input/deepfaketimit",
    "celeb-df-v2": "/kaggle/input/celeb-df-v2",
    "asvpoof-2019": "/kaggle/input/asvpoof-2019-dataset-la",
    "in-the-wild": "/kaggle/input/in-the-wild-audio-deepfake",
    "wavefake": "/kaggle/input/wavefake",
}

def extract_if_needed(name: str, src: str, dst: Path):
    """Extract zip/tar if not already done."""
    marker = dst / ".extracted"
    if marker.exists():
        print(f"  {name}: already extracted, skipping")
        return
    
    src_path = Path(src)
    if not src_path.exists():
        print(f"  {name}: mount path not found at {src}")
        return
    
    # Find archives
    archives = list(src_path.rglob("*.zip")) + list(src_path.rglob("*.tar.gz")) + list(src_path.rglob("*.tar"))
    if not archives:
        # Maybe already extracted in mount
        print(f"  {name}: no archives found, using mount directly")
        marker.touch()
        return
    
    for arch in archives:
        print(f"  {name}: extracting {arch.name}...", end=" ")
        try:
            if arch.suffix == ".zip":
                with zipfile.ZipFile(arch) as zf:
                    zf.extractall(dst)
            elif ".tar" in arch.suffixes:
                with tarfile.open(arch) as tf:
                    tf.extractall(dst)
            print("OK")
        except Exception as e:
            print(f"FAILED: {e}")
    
    marker.touch()

print("Extracting datasets...")
for name, src in DATASETS.items():
    dst = DATA_DIR / name
    dst.mkdir(exist_ok=True)
    extract_if_needed(name, src, dst)

print("\nData ready.")

In [ ]:
# Cell 5: Build manifests from extracted data
# This runs the existing build_manifest.py for FakeAVCeleb
# Other datasets need their own converters (see scripts/)

import subprocess

FAKEAVCELEB_DIR = DATA_DIR / "fakeavceleb"
MANIFEST_DIR = Path(REPO_DIR) / "src" / "data" / "manifests"
SPLIT_DIR = Path(REPO_DIR) / "src" / "data" / "splits"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

# Check if FakeAVCeleb has the expected structure
fakeav_root = None
for candidate in [FAKEAVCELEB_DIR, FAKEAVCELEB_DIR / "FakeAVCeleb_v1.2"]:
    if candidate.exists() and any((candidate / q).exists() for q in ["RealVideo-RealAudio", "FakeVideo-FakeAudio"]):
        fakeav_root = candidate
        break

if fakeav_root:
    print(f"Found FakeAVCeleb at: {fakeav_root}")
    !cd {REPO_DIR} && python scripts/build_manifest.py \
        --root {fakeav_root} \
        --out {MANIFEST_DIR}/fakeavceleb.jsonl \
        --splits-dir {SPLIT_DIR}/fakeavceleb --seed 42
else:
    print(f"FakeAVCeleb not found. Check mount paths.")
    print(f"Looking for: RealVideo-RealAudio/, FakeVideo-FakeAudio/ subdirs")

In [ ]:
# Cell 6: Configure training
import yaml

CONFIG = {
    "run_id": "run_001",  # CHANGE for each new run
    
    # Model
    "d_model": 768,
    "n_heads": 8,
    "n_fusion_layers": 4,
    "dropout": 0.1,
    "use_sync": True,
    "use_disentangle": True,
    "compose_quadrant": False,
    
    # Backbones (use fallback for T4 VRAM)
    "video_backbone": "fallback",
    "audio_backbone": "fallback",
    "video_model_name": "MCG-NJU/videomae-base",
    "audio_model_name": "microsoft/wavlm-base-plus",
    "freeze_blocks": 6,
    "freeze_feature_extractor": True,
    
    # Data
    "n_frames": 16,
    "audio_len": 64000,
    "shard_root": None,
    "feature_cache": None,
    "train_manifest": str(MANIFEST_DIR / "fakeavceleb.jsonl"),
    
    # Missing-modality
    "modality_dropout": 0.15,
    
    # Optimization (T4-friendly)
    "batch_size": 4,
    "num_workers": 2,
    "epochs": 30,
    "lr": 1.0e-4,
    "weight_decay": 1.0e-4,
    "log_every": 10,
    "out_dir": str(WORKING / "runs"),
    "local_dir": str(WORKING),
    
    # Loss weights
    "loss_weights": {"v": 1.0, "a": 1.0, "quad": 0.5, "loc": 0.5, "sync": 0.3, "disentangle": 0.1},
    
    "seed": 42,
}

# Save config
config_path = WORKING / "train_config.yaml"
with open(config_path, "w") as f:
    yaml.dump(CONFIG, f)

print(f"Config saved: {config_path}")
print(f"Run ID: {CONFIG['run_id']}")
print(f"Manifest: {CONFIG['train_manifest']}")

In [ ]:
# Cell 7: Train!
import sys
sys.path.insert(0, REPO_DIR)

# Set HF token in env for the backup module
os.environ["HF_TOKEN"] = hf_token

!cd {REPO_DIR} && python -m src.training.train \
    --config {config_path} \
    --run-id {CONFIG['run_id']}

In [ ]:
# Cell 8: Verify HF backup
from huggingface_hub import HfApi
api = HfApi(token=hf_token)

repo_id = "MoshinAli/david-net-av-backup"
try:
    files = list(api.list_repo_tree(repo_id, path_in_repo=f"runs/{CONFIG['run_id']}",
                                     repo_type="model", recursive=True))
    print(f"Files in HF repo for run {CONFIG['run_id']}:")
    for f in files:
        if hasattr(f, 'path'):
            print(f"  {f.path}")
except Exception as e:
    print(f"Could not list HF repo: {e}")
    print("This is OK if the run was short or training didn't complete.")